In [4]:
from pydantic import BaseModel,Field,computed_field
from typing import List
from enum import Enum

In [5]:
class JobType(str,Enum):
    REMOTE = 'remote'
    HYBRID = 'hybrid'
    ONSITE = 'onsite'

class ExperienceLevel(str, Enum):
    Internship = "Internship"
    Entry_level = "Entry_level"
    Associate = "Associate"
    Mid_Senior_level = "Mid_Senior_level"
    Director = "Director"
    Executive = "Executive"

In [6]:
class JobInfo(BaseModel):
    title : str | None = Field(default=None,description= "Primary job title or role to search for.This represents the main occupation or position of interest." ,examples=['AI engineer','Data Scientist','SQL','Java','Software Engineer'])
    location : str | None = Field(default=None,description= "The name of the country where job needs to be find, If any city name is entered then think of the contry in which the city exist",examples=['India','America'])
    days : int | None = Field(default=7,description= "Job posted within the last days ",examples=[1,3,7,14])
    companyName : List[str] | None = Field(default=None,description= "The List of companies which needs to consider first or which is only needs to considered, ordered by priority ",examples=['Google','Microsoft'])
    companyId : List[str] | None = Field(default=None,description= "The List of Ids of the companies which needs to consider first or which is only needs to considered, ordered by priority ",examples=['21345','5567483'])
    skipJobId : List[str] | None = Field(default=None,description= "The List of Ids of the companies which needs to  be skiped or not considered",examples=['21345','5567483'])
    jobType : List[JobType] | None = Field(default=None,description= "This is the list of type of job the user preferred, ordered by priority ",examples=[['remote','hybrid'],['onsite']])
    experience_level : List[ExperienceLevel] | None = Field(default=None,description= "Preferred experience levels for the job, ordered by priority (from most to least preferred).")
    # typeOfContract : List[str] | None = Field(default=None,description= "The type of the ontract preffered by the user rankwise ",examples=[['Part_time','Full_time']])
    limit : int | None = Field(default=3,description= "The number of jobs the user wants to find even if the user will say a big number limit it up to 3",le=3,ge=1,examples=[3,1,2])
    
    @computed_field
    @property
    def datePosted(self) -> str:
        return "r" + str(self.days * 86400)

    @computed_field
    @property
    def remote(self) -> List[int] | None:
        if not self.jobType:
            return None

        mapping = {
            JobType.ONSITE: '1',
            JobType.REMOTE: '2',
            JobType.HYBRID: '3',
        }

        return [mapping[jt] for jt in self.jobType]
    
    @computed_field
    @property
    def experienceLevel(self) -> List[str] | None:
        if not self.experience_level:
            return None

        mapping = {
            ExperienceLevel.Internship: "1",
            ExperienceLevel.Entry_level: "2",
            ExperienceLevel.Associate: "3",
            ExperienceLevel.Mid_Senior_level: "4",
            ExperienceLevel.Director: "5",
            ExperienceLevel.Executive: "6",
        }

        return [mapping[ctype] for ctype in self.experience_level]  
    
    

In [7]:

from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3.1:8b",
    temperature=0.2
)

In [8]:
# from langchain_google_genai import ChatGoogleGenerativeAI
# from dotenv import load_dotenv

# load_dotenv()

# llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [9]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(template="""
        You are job searching helping agent you need to pass the 
        data to job searchng API for that you need to structure the 
        output from the given user prompt : {user_prompt}
        """,
        input_variables=['user_prompt'])

In [10]:
import time
time0 = time.time()
print(time0)

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
time1 = time.time()
print("time1",time1-time0)
parser = PydanticOutputParser(pydantic_object=JobInfo)

prompt = PromptTemplate(
    template="""
You are a job-search helper agent.

Extract structured information from the user query.

{format_instructions}

User query:
{user_prompt}
""",
    input_variables=["user_prompt"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

time2 = time.time()
print("time2",time2-time1)

llm = OllamaLLM(model="llama3.1:8b", temperature=0)

chain = prompt | llm | parser

result = chain.invoke({
    "user_prompt": "Looking for a backend Java developer job in Berlin with Spring and 2+ years experience"
})

time3 = time.time()
print("time3",time3-time2)
print(result)
print(type(result))


1770460254.9491591
time1 0.002000093460083008
time2 0.004556417465209961
time3 69.3295533657074
title='Backend Java Developer' location='Berlin' days=7 companyName=None companyId=None skipJobId=None jobType=[<JobType.REMOTE: 'remote'>, <JobType.HYBRID: 'hybrid'>] experience_level=[<ExperienceLevel.Mid_Senior_level: 'Mid_Senior_level'>] limit=3 datePosted='r604800' remote=['2', '3'] experienceLevel=['4']
<class '__main__.JobInfo'>


c:\Users\DELL\miniconda3\envs\MCP\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [field_name='remote', input_value='2', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [field_name='remote', input_value='3', input_type=str])
  return self.__pydantic_serializer__.to_python(


In [11]:
from pydantic import Field,BaseModel
from typing import List
from langchain_core.prompts import PromptTemplate
# from langchain_google_genai import ChatGoogleGenerativeAI
# from dotenv import load_dotenv

In [12]:
# job_description = data[1]['description']
# Fetched 3 items
# {'id': '4347051914', 'url': 'https://www.linkedin.com/jobs/view/machine-learning-engineer-intern-at-upstart-4347051914', 'title': 'Machine Learning Engineer Intern', 'location': 'United States', 'companyName': 'Upstart', 'companyUrl': 'https://www.linkedin.com/company/upstart-network', 'recruiterName': '', 'recruiterUrl': '', 'experienceLevel': 'Internship', 'contractType': 'Internship', 'workType': 'Engineering and Information Technology', 'sector': 'Financial Services', 'salary': '$141,000.00/yr - $150,000.00/yr', 'applyType': 'EXTERNAL', 'applyUrl': '', 'postedTimeAgo': '3 days ago', 'postedDate': '2025-12-13T00:00:00.000Z', 'applicationsCount': 'Over 200 applicants', 'description': "About UpstartUpstart is the leading AI lending marketplace partnering with banks and credit unions to expand access to affordable credit. By leveraging Upstart's AI marketplace, Upstart-powered banks and credit unions can have higher approval rates and lower loss rates across races, ages, and genders, while simultaneously delivering the exceptional digital-first lending experience their customers demand. More than 80% of borrowers are approved instantly, with zero documentation to upload.Upstart is a digital-first company, which means that most Upstarters live and work anywhere in the United States. However, we also have offices in San Mateo, California; Columbus, Ohio; Austin, Texas; and New York City, NY (opening Summer 2026).Most Upstarters join us because they connect with our mission of enabling access to effortless credit based on true risk. If you are energized by the impact you can make at Upstart, we’d love to hear from you!The TeamMachine Learning is at the heart of Upstart’s business model, our models are the product. Our team includes research scientists, data scientists, and machine learning engineers who build and improve production models and the systems around them across the funnel: underwriting and pricing, fraud detection, performance marketing, loan servicing and fair lending/explainability. We tackle high‑impact problems, from underwriting and pricing to monitoring and fairness, where creativity, rigor, and strong engineering directly move the business.The RoleAs a Machine Learning Engineering Intern at Upstart, you’ll build tools and production‑grade code that make our models better and our researchers faster (think MLE: {Code} → {Better Code}). You will implement algorithmic improvements that boost predictive power, training efficiency, or serving latency; design reliable data/feature pipelines; and automate repeatable workflows that reduce time from research to deployment and monitoring. You’ll receive mentorship from experienced ML practitioners and collaborate closely with ML scientists to deliver solutions to real‑world engineering problems.How you’ll make an impact:Deliver projects that improve model performance, efficiency, latency or reliability.Write production-grade code, i.e. tested, reviewed and scalable Python.Communicate findings clearly to get buy-in for recommended next steps. Work with your mentor toalign stakeholders, and drive next steps.Minimum QualificationsStrong academic credentials with an ongoing bachelor’s or master’s degree in computer science, physics, machine learning, or other quantitative areas of study. We require that you are on track to graduate by the summer of 2027 (our internship program is not open to first- and second-year undergraduates). Programming skills in Python.Proficiency across machine learning, numerical computing, and software engineering fundamentals. Foundations in probability and statistics.Strong sense of intellectual curiosity, humility, drive and teamwork, as well as communication skills.Preferred QualificationsPhD studies or post-doctoral research in computer science, physics, machine learning, or other another quantitative field. If you are a PhD student, we require that you are on track to graduate by the summer of 2027. Experience building models and conducting statistical or quantitative research.Experience building ML tooling or solving real‑world ML engineering problems in industry (e.g., data pipelines, evaluation frameworks, training/serving), such as through prior internships. Position location This role is available in the following locations: RemoteTime zone requirements The team operates on the East/West coast time zones.Travel requirements As a digital first company, the majority of your work can be accomplished remotely. The majority of our employees can live and work anywhere in the U.S but are encouraged to to still spend high quality time in-person collaborating via regular onsites. The in-person sessions’ cadence varies depending on the team and role; most teams meet once or twice per quarter for 2-4 consecutive days at a time.At Upstart, your base pay is one part of your total compensation package. The anticipated base salary for this position is expected to be within the below range. Your actual base pay will depend on your geographic location–with our “digital first” philosophy, Upstart uses compensation regions that vary depending on location. Individual pay is also determined by job-related skills, experience, and relevant education or training. Your recruiter can share more about the specific salary range for your preferred location during the hiring process.In addition, Upstart provides employees with target bonuses, equity compensation, and generous benefits packages (including medical, dental, vision, and 401k).United States | Remote - Anticipated Base Salary Range$141,000—$150,000 USDUpstart is a proud Equal Opportunity Employer. We are dedicated to ensuring that underrepresented classes receive better access to affordable credit, and are just as committed to embracing diversity and inclusion in our hiring practices. We celebrate all cultures, backgrounds, perspectives, and experiences, and know that we can only become better together.If you require reasonable accommodation in completing an application, interviewing, completing any pre-employment testing, or otherwise participating in the employee selection process, please email candidate_accommodations@upstart.comhttps://www.upstart.com/candidate_privacy_policy", 'descriptionHtml': '<strong>About Upstart<br><br></strong>Upstart is the leading AI lending marketplace partnering with banks and credit unions to expand access to affordable credit. By leveraging Upstart&apos;s AI marketplace, Upstart-powered banks and credit unions can have higher approval rates and lower loss rates across races, ages, and genders, while simultaneously delivering the exceptional digital-first lending experience their customers demand. More than 80% of borrowers are approved instantly, with zero documentation to upload.<br><br>Upstart is a digital-first company, which means that most Upstarters live and work anywhere in the United States. However, we also have offices in San Mateo, California; Columbus, Ohio; Austin, Texas; and New York City, NY (opening Summer 2026).<br><br>Most Upstarters join us because they connect with our mission of enabling access to effortless credit based on true risk. If you are energized by the impact you can make at Upstart, we&#x2019;d love to hear from you!<br><br><strong>The Team<br><br></strong>Machine Learning is at the heart of Upstart&#x2019;s business model, our models <em>are</em> the product. Our team includes research scientists, data scientists, and machine learning engineers who build and improve production models and the systems around them across the funnel: underwriting and pricing, fraud detection, performance marketing, loan servicing and fair lending/explainability. We tackle high&#x2011;impact problems, from underwriting and pricing to monitoring and fairness, where creativity, rigor, and strong engineering directly move the business.<br><br><strong>The Role<br><br></strong>As a Machine Learning Engineering Intern at Upstart, you&#x2019;ll build tools and production&#x2011;grade code that make our models better and our researchers faster (think MLE: {Code} &#x2192; {Better Code}). You will implement algorithmic improvements that boost predictive power, training efficiency, or serving latency; design reliable data/feature pipelines; and automate repeatable workflows that reduce time from research to deployment and monitoring. You&#x2019;ll receive mentorship from experienced ML practitioners and collaborate closely with ML scientists to deliver solutions to real&#x2011;world engineering problems.<br><br><strong>How you&#x2019;ll make an impact:<br><br></strong><ul><li>Deliver projects that improve model performance, efficiency, latency or reliability.</li><li>Write production-grade code, i.e. tested, reviewed and scalable Python.</li><li>Communicate findings clearly to get buy-in for recommended next steps. Work with your mentor toalign stakeholders, and drive next steps.<br><br></li></ul><strong>Minimum Qualifications<br><br></strong><ul><li>Strong academic credentials with an ongoing bachelor&#x2019;s or master&#x2019;s degree in computer science, physics, machine learning, or other quantitative areas of study. We require that you are on track to graduate by the summer of 2027 (our internship program is not open to first- and second-year undergraduates). </li><li>Programming skills in Python.</li><li>Proficiency across machine learning, numerical computing, and software engineering fundamentals. Foundations in probability and statistics.</li><li>Strong sense of intellectual curiosity, humility, drive and teamwork, as well as communication skills.<br><br></li></ul><strong>Preferred Qualifications<br><br></strong><ul><li>PhD studies or post-doctoral research in computer science, physics, machine learning, or other another quantitative field. If you are a PhD student, we require that you are on track to graduate by the summer of 2027. </li><li>Experience building models and conducting statistical or quantitative research.</li><li>Experience building ML tooling or solving real&#x2011;world ML engineering problems in industry (e.g., data pipelines, evaluation frameworks, training/serving), such as through prior internships. <br><br></li></ul><strong>Position location</strong> This role is available in the following locations: Remote<br><br><strong>Time zone requirements</strong> The team operates on the East/West coast time zones.<br><br><strong>Travel requirements</strong> As a digital first company, the majority of your work can be accomplished remotely. The majority of our employees can live and work anywhere in the U.S but are encouraged to to still spend high quality time in-person collaborating via regular onsites. The in-person sessions&#x2019; cadence varies depending on the team and role; most teams meet once or twice per quarter for 2-4 consecutive days at a time.<br><br>At Upstart, your base pay is one part of your total compensation package. The anticipated base salary for this position is expected to be within the below range. Your actual base pay will depend on your geographic location&#x2013;with our &#x201c;digital first&#x201d; philosophy, Upstart uses compensation regions that vary depending on location. Individual pay is also determined by job-related skills, experience, and relevant education or training. Your recruiter can share more about the specific salary range for your preferred location during the hiring process.<br><br>In addition, Upstart provides employees with target bonuses, equity compensation, and generous benefits packages (including medical, dental, vision, and 401k).<br><br>United States | Remote - Anticipated Base Salary Range<br><br>&#x24;141,000&#x2014;&#x24;150,000 USD<br><br>Upstart is a proud Equal Opportunity Employer. We are dedicated to ensuring that underrepresented classes receive better access to affordable credit, and are just as committed to embracing diversity and inclusion in our hiring practices. We celebrate all cultures, backgrounds, perspectives, and experiences, and know that we can only become better together.<br><br><em>If you require reasonable accommodation in completing an application, interviewing, completing any pre-employment testing, or otherwise participating in the employee selection process, please email </em><em>candidate_accommodations@upstart.com<br><br></em>https://www.upstart.com/candidate_privacy_policy<br><br>'}

In [13]:
job_description = """About UpstartUpstart is the leading AI lending marketplace partnering with banks and credit unions to expand access to affordable credit. By leveraging Upstart's AI marketplace, Upstart-powered banks and credit unions can have higher approval rates and lower loss rates across races, ages, and genders, while simultaneously delivering the exceptional digital-first lending experience their customers demand. More than 80% of borrowers are approved instantly, with zero documentation to upload.Upstart is a digital-first company, which means that most Upstarters live and work anywhere in the United States. However, we also have offices in San Mateo, California; Columbus, Ohio; Austin, Texas; and New York City, NY (opening Summer 2026).Most Upstarters join us because they connect with our mission of enabling access to effortless credit based on true risk. If you are energized by the impact you can make at Upstart, we’d love to hear from you!The TeamMachine Learning is at the heart of Upstart’s business model, our models are the product. Our team includes research scientists, data scientists, and machine learning engineers who build and improve production models and the systems around them across the funnel: underwriting and pricing, fraud detection, performance marketing, loan servicing and fair lending/explainability. We tackle high‑impact problems, from underwriting and pricing to monitoring and fairness, where creativity, rigor, and strong engineering directly move the business.The RoleAs a Machine Learning Engineering Intern at Upstart, you’ll build tools and production‑grade code that make our models better and our researchers faster (think MLE: {Code} → {Better Code}). You will implement algorithmic improvements that boost predictive power, training efficiency, or serving latency; design reliable data/feature pipelines; and automate repeatable workflows that reduce time from research to deployment and monitoring. You’ll receive mentorship from experienced ML practitioners and collaborate closely with ML scientists to deliver solutions to real‑world engineering problems.How you’ll make an impact:Deliver projects that improve model performance, efficiency, latency or reliability.Write production-grade code, i.e. tested, reviewed and scalable Python.Communicate findings clearly to get buy-in for recommended next steps. Work with your mentor toalign stakeholders, and drive next steps.Minimum QualificationsStrong academic credentials with an ongoing bachelor’s or master’s degree in computer science, physics, machine learning, or other quantitative areas of study. We require that you are on track to graduate by the summer of 2027 (our internship program is not open to first- and second-year undergraduates). Programming skills in Python.Proficiency across machine learning, numerical computing, and software engineering fundamentals. Foundations in probability and statistics.Strong sense of intellectual curiosity, humility, drive and teamwork, as well as communication skills.Preferred QualificationsPhD studies or post-doctoral research in computer science, physics, machine learning, or other another quantitative field. If you are a PhD student, we require that you are on track to graduate by the summer of 2027. Experience building models and conducting statistical or quantitative research.Experience building ML tooling or solving real‑world ML engineering problems in industry (e.g., data pipelines, evaluation frameworks, training/serving), such as through prior internships. Position location This role is available in the following locations: RemoteTime zone requirements The team operates on the East/West coast time zones.Travel requirements As a digital first company, the majority of your work can be accomplished remotely. The majority of our employees can live and work anywhere in the U.S but are encouraged to to still spend high quality time in-person collaborating via regular onsites. The in-person sessions’ cadence varies depending on the team and role; most teams meet once or twice per quarter for 2-4 consecutive days at a time.At Upstart, your base pay is one part of your total compensation package. The anticipated base salary for this position is expected to be within the below range. Your actual base pay will depend on your geographic location–with our “digital first” philosophy, Upstart uses compensation regions that vary depending on location. Individual pay is also determined by job-related skills, experience, and relevant education or training. Your recruiter can share more about the specific salary range for your preferred location during the hiring process.In addition, Upstart provides employees with target bonuses, equity compensation, and generous benefits packages (including medical, dental, vision, and 401k).United States | Remote - Anticipated Base Salary Range$141,000—$150,000 USDUpstart is a proud Equal Opportunity Employer. We are dedicated to ensuring that underrepresented classes receive better access to affordable credit, and are just as committed to embracing diversity and inclusion in our hiring practices. We celebrate all cultures, backgrounds, perspectives, and experiences, and know that we can only become better together.If you require reasonable accommodation in completing an application, interviewing, completing any pre-employment testing, or otherwise participating in the employee selection process, please email candidate_accommodations@upstart.comhttps://www.upstart.com/candidate_privacy_policy", 'descriptionHtml': '<strong>About Upstart<br><br></strong>Upstart is the leading AI lending marketplace partnering with banks and credit unions to expand access to affordable credit. By leveraging Upstart&apos;s AI marketplace, Upstart-powered banks and credit unions can have higher approval rates and lower loss rates across races, ages, and genders, while simultaneously delivering the exceptional digital-first lending experience their customers demand. More than 80% of borrowers are approved instantly, with zero documentation to upload.<br><br>Upstart is a digital-first company, which means that most Upstarters live and work anywhere in the United States. However, we also have offices in San Mateo, California; Columbus, Ohio; Austin, Texas; and New York City, NY (opening Summer 2026).<br><br>Most Upstarters join us because they connect with our mission of enabling access to effortless credit based on true risk. If you are energized by the impact you can make at Upstart, we&#x2019;d love to hear from you!<br><br><strong>The Team<br><br></strong>Machine Learning is at the heart of Upstart&#x2019;s business model, our models <em>are</em> the product. Our team includes research scientists, data scientists, and machine learning engineers who build and improve production models and the systems around them across the funnel: underwriting and pricing, fraud detection, performance marketing, loan servicing and fair lending/explainability. We tackle high&#x2011;impact problems, from underwriting and pricing to monitoring and fairness, where creativity, rigor, and strong engineering directly move the business.<br><br><strong>The Role<br><br></strong>As a Machine Learning Engineering Intern at Upstart, you&#x2019;ll build tools and production&#x2011;grade code that make our models better and our researchers faster (think MLE: {Code} &#x2192; {Better Code}). You will implement algorithmic improvements that boost predictive power, training efficiency, or serving latency; design reliable data/feature pipelines; and automate repeatable workflows that reduce time from research to deployment and monitoring. You&#x2019;ll receive mentorship from experienced ML practitioners and collaborate closely with ML scientists to deliver solutions to real&#x2011;world engineering problems.<br><br><strong>How you&#x2019;ll make an impact:<br><br></strong><ul><li>Deliver projects that improve model performance, efficiency, latency or reliability.</li><li>Write production-grade code, i.e. tested, reviewed and scalable Python.</li><li>Communicate findings clearly to get buy-in for recommended next steps. Work with your mentor toalign stakeholders, and drive next steps.<br><br></li></ul><strong>Minimum Qualifications<br><br></strong><ul><li>Strong academic credentials with an ongoing bachelor&#x2019;s or master&#x2019;s degree in computer science, physics, machine learning, or other quantitative areas of study. We require that you are on track to graduate by the summer of 2027 (our internship program is not open to first- and second-year undergraduates). </li><li>Programming skills in Python.</li><li>Proficiency across machine learning, numerical computing, and software engineering fundamentals. Foundations in probability and statistics.</li><li>Strong sense of intellectual curiosity, humility, drive and teamwork, as well as communication skills.<br><br></li></ul><strong>Preferred Qualifications<br><br></strong><ul><li>PhD studies or post-doctoral research in computer science, physics, machine learning, or other another quantitative field. If you are a PhD student, we require that you are on track to graduate by the summer of 2027. </li><li>Experience building models and conducting statistical or quantitative research.</li><li>Experience building ML tooling or solving real&#x2011;world ML engineering problems in industry (e.g., data pipelines, evaluation frameworks, training/serving), such as through prior internships."""

In [14]:

class Resume(BaseModel):
    skills : List[str] = Field(default=['No skills'], description="The main programing and technikal skills focus more on the domain specific skills rather than soft skills")
    Profile : str = Field(default='No profile', description="The brief info about the user found in the resume text")
    Projects : List[str] = Field(defaul=['No Projects'], description="The Projects that are built by the user found in the resume")
    Certifications : List[str] = Field(default=['User have No certifications'], description='THe certifications of the user in the resume')
    Experience : List[str] = Field(default=['User Have No experience'], description="The Experience of the user mentioned in the resume")
    Education : List[str] = Field(default='[No Education]',description='The education oof the yser found in the resume')

class similar_and_feadback(BaseModel):
    similarity : int = Field(...,le = 100,ge=0,description='Give the Similarity score between 0 to 100 on well the job is maching the user resume') 
    feadback   : str = Field(default = 'No feadback', description='Tell him what is missing from the user resume to get this job, kind of the feadback, what sector is lacking and what is lacking')

class Job_Description(BaseModel):
    skills : List[str] = Field(..., description="Extract the skills and what things are important for the Job from the job description") 
    
class Job_Summary(BaseModel):
    skills : List[str] = Field(...,description= 'Extract the skills that are mentioned in the Job description')
    job_info : str = Field(...,description = "Summary of the job in 3 lines")

E:\Temp\ipykernel_21720\585651287.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'defaul'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  Projects : List[str] = Field(defaul=['No Projects'], description="The Projects that are built by the user found in the resume")


In [15]:
parser = PydanticOutputParser(pydantic_object=Job_Summary)
job_description_template = PromptTemplate(
    template="""You are a job-description summarizer including all the importand points like roles, position, skills required agent.

Extract structured information from the job description.
{format_instruction} 

job_description: 
{job_description}""",
input_variables=['job_description'], partial_variables={
    "format_instruction":parser.get_format_instructions()
    }
)

In [16]:
job_summary_agent = job_description_template | llm | parser

In [17]:
job_summary_output = job_summary_agent.invoke({'job_description':job_description})

In [18]:
job_summary_output

Job_Summary(skills=['Machine Learning', 'Python', 'Probability and Statistics', 'Numerical Computing', 'Software Engineering'], job_info="As a Machine Learning Engineering Intern at Upstart, you'll build tools and production-grade code that make our models better and our researchers faster. You will implement algorithmic improvements that boost predictive power, training efficiency, or serving latency; design reliable data/feature pipelines; and automate repeatable workflows that reduce time from research to deployment and monitoring.")

In [19]:
resume_text = """Profile
Phone: 8866562577
G-mail: jugallachhwnai15@gmail.com
Github: https://github.com/Jugal-lachhwani
Address: 31, Sagar Society, Uttamnagar,
Maninagar,Ahmedabad
B-Tech in CSE + AIML
Charusat
Data Structures & Algorithms  (DSA) : 
Array, Strings, Tree, Graphs 
TOOLS & FRAMEWORKS: GIT, SQL,
NUMPY, PANDAS, MATPLOTLIB , ,SPACY
Programming Languages: Python, 
SQL C++,
Machine Learning & Deep Learning:
TensorFlow, PyTorch, Scikit-learn, NLP
Education
Programming Skills
English
Hindi
Language
Certification
JUGAL LACHHWANI
2. Personalized Tutor for Students
An chatbot made specially for gate
students by using Gemini By
implementing the Gate knowledge i
3 . Fake News Detection using Machine Learning
Working with the wider development
team.
Manage website design, content, and SEO
Marketing, Branding and Logo Design
Enthusiastic and self-motivated Computer Science
student with a strong foundation in Data Structures &
Algorithms, Machine Learning, and Deep Learning.
Passionate about problem-solving, competitive
programming, and developing AI-driven applications.
Adept at building scalable projects, working with tools
and library like Python, Pandas, Matplotlib, Seaborn and
SQL. Seeking opportunities to apply technical skills in
innovative projects and collaborative environments.
NPTEL-
Programming
Data Structures
and Algorithms
using Python
Projects
1.  Password Checker and Generator using C++
Hackkerrank-
JAVA Basic
Soft Skills
Strong analytical thinking, effective
communication, teamwork,
adaptability, and problem-solving.
AI-Ml Student
This is project made with C++ for checking
and generating a strng password
LeetCode:
https://leetcode.com/u/Jugallachhwani/"""

In [20]:
resume_parser = PydanticOutputParser(pydantic_object=Resume)
RESUME = PromptTemplate(template="""
        You are thoughtful agent help in extracting the Important features 
        from the resume text. Resume text is not structures as it is extracted 
        from a knowledge so use your llm skills to extract the following:
        Skills : The main programing and technikal skills focus more on the domain specific skills rather than soft skills
        Profile : The brief info about the user found in the resume text
        Projects : The Projects that are built by the user found in the resume
        Certifications : THe certifications of the user in the resume
        Experience : The Experience of the user mentioned in the resume
        Education : The education oof the yser found in the resume
        extract the fields using the following format_instruction from RESUME text: {format_instruction}
        The Remuse text is : {resume_text}
""", input_variables="resume_text", partial_variables= {"format_instruction" : resume_parser.get_format_instructions()})


resume_feedback_parser = PydanticOutputParser(pydantic_object=similar_and_feadback)
RESUME_FEADBACK = PromptTemplate(template = """
        You are a agent that hepls users to find the best job be hepling them improve there resume
        Your job is to tell what are the things that lack In there resume.
        What are those keywords, skills or projects what will help him to improve the resume,
        Along with that give the similarity score between 0 to 100 how well the jon suits the user
        Give the output in the specific format instructions:
        {format_instructions}
        Resemue skills : {resume_skills}
        Job Info : {job_info}
        Job skills : {job_skills}
        resume Profile : {resume_profile}
""", input_variables=["resume_skills","job_info","job_skills","resume_profile"],
partial_variables={"format_instructions":resume_feedback_parser.get_format_instructions()}
)

In [21]:
# prompt = PromptTemplate(template=RESUME,
#         input_variables=['resume_text'])

In [22]:
from pydantic import BaseModel, Field
from typing import List

class Resume(BaseModel):
    skills: List[str] = Field(
        default_factory=lambda: ["No skills"]
    )
    profile: str = Field(
        default="No profile"
    )
    projects: List[str] = Field(
        default_factory=lambda: ["No projects"]
    )
    certifications: List[str] = Field(
        default_factory=lambda: ["No certifications"]
    )
    experience: List[str] = Field(
        default_factory=lambda: ["No experience"]
    )
    education: List[str] = Field(
        default_factory=lambda: ["No education"]
    )


In [23]:
resume_parser = PydanticOutputParser(pydantic_object=Resume)

RESUME = PromptTemplate(
    template="""
You are an information extraction agent.

Extract the following fields from the resume text and return ONLY a JSON object
that matches the schema exactly.

{format_instructions}

Resume text:
{resume_text}
""",
    input_variables=["resume_text"],
    partial_variables={
        "format_instructions": resume_parser.get_format_instructions()
    }
)


In [25]:
resume_agent = RESUME | llm | resume_parser

resume_output = resume_agent.invoke({
    "resume_text": resume_text
})

print(resume_output)

skills=['GIT', 'SQL', 'NUMPY', 'PANDAS', 'MATPLOTLIB', 'SPACY', 'Python', 'SQL C++', 'TensorFlow', 'PyTorch', 'Scikit-learn'] profile='Enthusiastic and self-motivated Computer Science student with a strong foundation in Data Structures & Algorithms, Machine Learning, and Deep Learning.' projects=['Password Checker and Generator using C++', 'Fake News Detection using Machine Learning', 'NPTEL-Programming Data Structures and Algorithms using Python'] certifications=[] experience=[] education=['B-Tech in CSE + AIML at Charusat']


In [26]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import PydanticOutputParser


llm = OllamaLLM(
    model="llama3.1:8b",
    temperature=0.2
)
class SimilarityAndFeedback(BaseModel):
    similarity: int = Field(
        ...,
        ge=0,
        le=100,
        description="Similarity score between 0 and 100 indicating how well the job matches the resume."
    )
    feedback: str = Field(
        default="No feedback",
        description="Actionable feedback describing missing skills, keywords, or projects needed to improve the resume for this job."
    )

resume_feedback_parser = PydanticOutputParser(pydantic_object=SimilarityAndFeedback)
RESUME_FEEDBACK = PromptTemplate(
    template="""
You are a resume improvement agent.

Analyze how well the resume matches the given job.

Tasks:
1. Identify missing skills, keywords, or projects.
2. Provide clear and actionable feedback.
3. Give a similarity score between 0 and 100.

Follow the format instructions STRICTLY.
Return only the structured output.

{format_instructions}

Resume skills:
{resume_skills}

Resume profile:
{resume_profile}

Job information:
{job_info}

Job required skills:
{job_skills}
""",
    input_variables=[
        "resume_skills",
        "resume_profile",
        "job_info",
        "job_skills"
    ],
    partial_variables={
        "format_instructions": resume_feedback_parser.get_format_instructions()
    }
)

chain = RESUME_FEEDBACK | llm | resume_feedback_parser

result = chain.invoke({
    "resume_skills": resume_output.skills,
    "job_info": job_summary_output.job_info,
    "job_skills": job_summary_output.skills,
    "resume_profile": resume_output.profile
})


In [27]:
result

SimilarityAndFeedback(similarity=80, feedback="The resume matches the job requirements well, but there are a few areas for improvement. \n    - The resume lacks keywords such as 'Probability and Statistics' which is crucial for Machine Learning.\n    - Although the resume mentions 'Python', it's essential to emphasize this skill more prominently since it's a primary requirement.\n    - The resume does not explicitly mention 'Software Engineering' skills, which are necessary for production-grade code development.\n    - Consider adding projects or experiences that demonstrate algorithmic improvements and reliable data/feature pipelines.")